In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# 1. Cargar los datasets
# (Asegúrate de colocar las rutas correctas de tus archivos)
df_clima = pd.read_csv("Consolidado_Clima_Zonas.csv")
df_principal = pd.read_csv("dataset_papas_con_diesel_hasta_abril_2026.csv")

In [ ]:
df_clima.isnull().sum()

,0
region,0
zona,0
FECHA,0
YEAR,0
DOY,0
T2M,0
T2M_MIN,0
T2M_MAX,0
IMERG_PRECTOT,0
dias_con_helada_en_cultivo,0


In [ ]:
df_principal.isnull().sum()

,0
fecha,0
volumen,0
variedad,0
zona,0
volumen_log,0
region,0
precio_promedio_kg,0
indice_precio_diesel,0


In [ ]:
df_clima["zona"].unique()

array(['Abancay', 'Andahuaylas', 'Antabamba', 'Arequipa', 'Camana',
       'Caraveli', 'Caylloma', 'Cangallo', 'Huamanga', 'Huanca Sancos',
       'Huanta', 'La Mar', 'Lucanas', 'Parinacochas', 'Vilcas Huaman',
       'Acobamba', 'Angaraes', 'Churcampa', 'Huancavelica', 'Huaytara',
       'Tayacaja', 'Ambo', 'Huamalies', 'Huanuco', 'Huaycabamba',
       'Pachitea', 'Chincha', 'Ica', 'Nazca', 'Palpa', 'Pisco',
       'Chanchamayo', 'Concepcion', 'Huancayo', 'Jauja', 'Junin', 'Tarma',
       'Yauli', 'Julcan', 'Sanchez Carrion', 'Santiago De Chuco',
       'Barranca', 'Cajatambo', 'Canete', 'Canta', 'Huaral', 'Huarochiri',
       'Huaura', 'Lima', 'Oyon', 'Yauyos', 'Daniel Alcides Carrion',
       'Oxapampa', 'Pasco'], dtype=object)

In [ ]:
df_principal["zona"].unique()

array(['Jauja', 'Huancayo', 'Huamanga', 'Arequipa', 'Pasco', 'Lima',
       'Huaral', 'Tarma', 'Junin', 'Huanuco', 'Ambo', 'Huancavelica',
       'Huanta', 'Andahuaylas', 'Daniel Alcides Carrion', 'Tayacaja',
       'Canta', 'Sanchez Carrion', 'Cangallo', 'Churcampa',
       'Santiago De Chuco', 'Barranca', 'Yauli', 'Huanca Sancos',
       'Acobamba', 'Nazca', 'Abancay', 'Huaura', 'Concepcion', 'Pachitea',
       'Ica', 'Camana', 'Lucanas', 'Canete', 'Pisco', 'Oxapampa',
       'Cajatambo', 'Huaytara', 'Huarochiri', 'Huamalies', 'Caylloma',
       'Yauyos', 'Chincha', 'Palpa', 'Parinacochas', 'Antabamba',
       'Huaycabamba', 'Caraveli', 'La Mar', 'Angaraes', 'Chanchamayo',
       'Julcan', 'Oyon'], dtype=object)

In [ ]:
set(df_clima["region"].unique())

{'Apurimac',
 'Arequipa',
 'Ayacucho',
 'Huancavelica',
 'Huanuco',
 'Ica',
 'Junin',
 'La Libertad',
 'Lima',
 'Pasco'}

In [ ]:
set(df_principal["region"].unique())

{'Apurimac',
 'Arequipa',
 'Ayacucho',
 'Huancavelica',
 'Huanuco',
 'Ica',
 'Junin',
 'La libertad',
 'Lima',
 'Pasco'}

In [ ]:

# 2. Homogeneizar tipos de datos y nombres para el cruce
# Pasamos las fechas a formato datetime para evitar fallos por texto o espacios
df_clima["FECHA"] = pd.to_datetime(df_clima["FECHA"])
df_principal["fecha"] = pd.to_datetime(df_principal["fecha"])

# Convertimos a minúsculas o texto limpio las columnas de cruce para evitar que "Jauja" y "Jauja " no crucen
for col in ["region", "zona"]:
    df_clima[col] = df_clima[col].astype(str).str.strip()
    df_principal[col] = df_principal[col].astype(str).str.strip()

for col in ["region", "zona"]:
    df_clima[col] = (
        df_clima[col]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    df_principal[col] = (
        df_principal[col]
        .astype(str)
        .str.strip()
        .str.lower()
    )

# 3. Limpieza de datos climáticos antes del merge
# Renombrar IMERG_PRECTOT a lluvia_hoy
df_clima = df_clima.rename(columns={"IMERG_PRECTOT": "lluvia_hoy"})

# Reemplazar el -999.0 (No Data de la NASA) por NaN o 0 (según prefieras, aquí lo dejamos como NaN)
df_clima["lluvia_hoy"] = df_clima["lluvia_hoy"].replace(-999.0, np.nan)

# Seleccionamos solo las columnas del clima que nos interesa llevar al principal
columnas_clima_interes = [
    "region",
    "zona",
    "FECHA",
    "lluvia_hoy",
    "dias_con_helada_en_cultivo",
    "T2M",
    "T2M_MIN",
    "T2M_MAX"
]
df_clima_filtrado = df_clima[columnas_clima_interes]

# 4. REALIZAR EL MERGE (Cruzar por Izquierda)
# Unimos por las tres llaves: fecha, region y zona
df_unido = pd.merge(
    df_principal,
    df_clima_filtrado,
    left_on=["fecha", "region", "zona"],
    right_on=["FECHA", "region", "zona"],
    how="left"
)

# Como la columna 'FECHA' (mayúscula) se duplica en el cruce, la eliminamos
df_unido = df_unido.drop(columns=["FECHA"])

# 5. Ordenar el dataset final (opcional, por consistencia: fecha más reciente primero)
df_unido = df_unido.sort_values(
    by=["fecha", "region", "zona"],
    ascending=[False, False, False]
).reset_index(drop=True)

df_unido["fecha"] = pd.to_datetime(df_unido["fecha"])

df_unido = df_unido[df_unido["fecha"] <= "2026-02-03"]

# Guardar el resultado final
df_unido.to_csv("Dataset_Final_Modelado.csv", index=False)

print("¡Merge completado con éxito!")
print(f"Dimensiones del dataset principal original: {df_principal.shape}")
print(f"Dimensiones del dataset final generado: {df_unido.shape}")
print("\nColumnas resultantes:")
print(df_unido.columns.tolist())

print("\nMuestra de los primeros datos combinados:")
print(df_unido.head())



¡Merge completado con éxito!
Dimensiones del dataset principal original: (39180, 8)
Dimensiones del dataset final generado: (37482, 13)

Columnas resultantes:
['fecha', 'volumen', 'variedad', 'zona', 'volumen_log', 'region', 'precio_promedio_kg', 'indice_precio_diesel', 'lluvia_hoy', 'dias_con_helada_en_cultivo', 'T2M', 'T2M_MIN', 'T2M_MAX']

Muestra de los primeros datos combinados:
          fecha  volumen     variedad                    zona  volumen_log  \
1698 2026-02-03     18.0  Papa Blanca                   pasco     2.944439   
1699 2026-02-03     10.0  Papa Blanca  daniel alcides carrion     2.397895   
1700 2026-02-03     60.0  Papa Blanca                    lima     4.110874   
1701 2026-02-03     19.0  Papa Blanca                  huaura     2.995732   
1702 2026-02-03     38.0   Papa Unica                  huaral     3.663562   

     region  precio_promedio_kg  indice_precio_diesel  lluvia_hoy  \
1698  pasco                0.94               88.4916       12.51   
1699  

In [ ]:
df_unido.isnull().sum()

,0
fecha,0
volumen,0
variedad,0
zona,0
volumen_log,0
region,0
precio_promedio_kg,0
indice_precio_diesel,0
lluvia_hoy,0
dias_con_helada_en_cultivo,0
